# HW6: Probing Grounding Effects in Vision-Language Models

This notebook demonstrates 5 different methods for object localization using CLIP:
1. GradCAM based on image-text similarity
2. Feature cosine similarity
3. Attention map with anchor patch
4. CLIP Surgery
    https://github.com/xmed-lab/CLIP_Surgery
5. GEM (Grounding Everything Module)
    https://github.com/WalBouss/GEM



In [ ]:
# TODO: Install the following packages, make sure the version of open_clip package is suitable for GEM requirements (e.g. open-clip-torch==2.23.0)
# TODO: You can start with creating a conda environment with python=3.10, and then install the packages in the requirements.txt file.

import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import open_clip
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


/home/fsyang/miniconda3/envs/HW6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fsyang/miniconda3/envs/HW6/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Using device: cpu


## Load CLIP Model from OpenAI (ViT-B/16)


In [ ]:
# Load CLIP ViT-B/16 model
model_name = 'ViT-B-16'
pretrained = 'openai'

# Load model and preprocessing
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    model_name, 
    pretrained=pretrained,
    device=device
)
clip_model.eval()

tokenizer = open_clip.get_tokenizer(model_name)

print("CLIP model loaded successfully!")


CLIP model loaded successfully!


## Get data and preprocess it 

In [ ]:
# Load and preprocess image


# Please don't change the given image-text pairs
img_dict = {
    "0": "./images/fire_hydrant_car.jpg",
    "1": "./images/jet_people.jpg",
    # Demo image
    "2": "./images/drink_donut.jpg", 

}
basic_caption_dict = {
    "0": "fire hydrant",
    "1": "jet",
    # Demo caption
    "2": "drink", 
}

# Set index to fetch a single image and its caption
data_index = "1"  # Change this index to select a different image

# You are required to use the following layer indices, getting information from the 3rd, 6th, 9th, and 12th transformer blocks
layer_indices = [2, 5, 8, 11]
patch_grid_size = 14

# Load one image and preprocess it
image = Image.open(img_dict[data_index]).convert('RGB')
image_tensor = preprocess(image).unsqueeze(0).to(device)

# Load one caption and tokenize
text_token = tokenizer([basic_caption_dict[data_index]]).to(device)

print(f"Image shape: {image_tensor.shape}")
print(f"Text shape: {text_token.shape}")





NameError: name 'Image' is not defined

In [ ]:
print("DATA INDEX NOW =", data_index)
print("DICT VALUE =", img_dict[data_index])

In [ ]:
# TODO: Extract any feature map from a given transformer block
def get_patch_features(model, image_tensor, block_idx=11):
    """
    Extract any feature map from block[block_idx] of ViT.
    Args:
        model: CLIP model
        image_tensor: preprocessed image tensor
        block_idx: index of the transformer block to extract from
    Returns:
        The feature map at the specified location inside the transformer block.
        (Project the feature map to mutual dimension for comparison with text feature)
    """
    visual = model.visual

    # TODO: Prepare visual transformer input

    # Patch embedding: conv1 converts image to patch embeddings
    x = visual.conv1(image_tensor)  # [B, embed_dim, grid_h, grid_w]
    x = x.reshape(x.shape[0], x.shape[1], -1)  # [B, embed_dim, num_patches]
    x = x.permute(0, 2, 1)  # [B, num_patches, embed_dim]
    
    # Add CLS token
    cls_token = visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
    x = torch.cat([cls_token, x], dim=1)  # [B, 1 + num_patches, embed_dim]
    
    # Add positional embedding
    x = x + visual.positional_embedding.to(x.dtype)
    
    # Pre-LayerNorm
    x = visual.ln_pre(x)
    
    # Convert to [seq_len, batch, embed_dim] for transformer
    x = x.permute(1, 0, 2)  # [seq_len, B, embed_dim]

    # Step through transformer blocks, hook out desired feature from chosen block
    feature_map = None
    for i, resblock in enumerate(visual.transformer.resblocks):
        # TODO: extract the feature map from the chosen block
        x = resblock(x)
        if i == block_idx:
            feature_map = x.clone()

    # TODO: Remove CLS token for patch features
    # Convert back to [B, seq_len, embed_dim]
    feature_map = feature_map.permute(1, 0, 2)  # [B, seq_len, embed_dim]

    # Step 3: Remove CLS token for patch features
    feature_map = feature_map[:, 1:, :]  # [B, num_patches, embed_dim]
    
    # Step 4: Apply LayerNorm
    feature_map = visual.ln_post(feature_map)

    # TODO: dim projection
    if visual.proj is not None:
        feature_map = feature_map @ visual.proj  # [B, num_patches, proj_dim]
    
    return feature_map


: 

## Method 1: GradCAM based on Image-Text Similarity


In [ ]:
# TODO: Generate GradCAM heatmap based on image-text similarity
def generate_gradcam(model, image_tensor, text_tokens, block_idx=11):
    """Generate GradCAM heatmap based on image-text similarity"""
    activations = None
    gradients = None

    def forward_hook(module, input, output):
        nonlocal activations
        activations = output

    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0] if grad_output[0] is not None else None

    # TODO: Register hooks on the assigned transformer block's layer norm (the normalizer before the attention)
    block = model.visual.transformer.resblocks[block_idx].ln_1
    handle_forward = block.register_forward_hook(forward_hook)
    handle_backward = block.register_full_backward_hook(backward_hook)

    # TODO: Forward pass
    image_features = model.encode_image(image_tensor)  # [B, embed_dim]
    text_features = model.encode_text(text_tokens)  # [B, embed_dim]
    
    # Normalize features
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)
    
    # TODO: Compute similarity
    similarity = (image_features * text_features).sum()

    # Backward pass
    model.zero_grad()
    similarity.backward()

    # TODO: Remove CLS token (first token)
    activations = activations[1:, :, :]  # [num_patches, B, embed_dim]
    gradients = gradients[1:, :, :]  # [num_patches, B, embed_dim]
    # Convert to [B, num_patches, embed_dim]
    activations = activations.permute(1, 0, 2)  # [B, num_patches, embed_dim]
    gradients = gradients.permute(1, 0, 2)  # [B, num_patches, embed_dim]
    
    # TODO: Compute weights per feature dimension (global average pooling over spatial patches)
    weights = gradients.mean(dim=1, keepdim=True)  # [B, 1, embed_dim]
    
    # TODO: Apply the weights to each feature dimension at each spatial patch
    cam = (weights * activations).sum(dim=-1)  # [B, num_patches]
    cam = F.relu(cam)  # Apply ReLU
    # Normalize to [0, 1]
    cam = cam.squeeze(0).cpu().detach().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    # Remove hooks
    handle_forward.remove()
    handle_backward.remove()
    
    return cam

# Run GradCAM for different layer indices
gradcam_heatmaps_resized = []
original_size = image.size

with torch.enable_grad():
    for idx in layer_indices:
        image_tensor_grad = image_tensor.clone().requires_grad_(True)
        cam_gradcam = generate_gradcam(clip_model, image_tensor_grad, text_token, block_idx=idx)
        # TODO: Reshape cam_gradcam to spatial dimensions
        cam_gradcam_2d = cam_gradcam.reshape(patch_grid_size, patch_grid_size)
        gradcam_heatmap_resized = cv2.resize(cam_gradcam_2d, original_size, interpolation=cv2.INTER_LINEAR)
        gradcam_heatmaps_resized.append(gradcam_heatmap_resized)

# Visualize all heatmaps together
fig, axes = plt.subplots(1, len(layer_indices) + 1, figsize=(6*(len(layer_indices)+1), 6), constrained_layout=True)

# Show original image first
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Show GradCAMs (This is just for visualization, not used in the final report)
for i, cam in enumerate(gradcam_heatmaps_resized):
    ax = axes[i+1]
    ax.imshow(image, alpha=0.5, aspect='equal')
    im = ax.imshow(cam, cmap='jet', alpha=0.5, aspect='equal')
    ax.set_title(f"GradCAM Heatmap\nLayer {layer_indices[i]}")
    ax.axis('off')
plt.colorbar(im, ax=ax)

plt.show()


## Method 2: Feature Cosine Similarity


In [ ]:
# TODO: Run Feature Cosine Similarity for different layer indices
original_size = image.size
similarity_heatmaps = []

with torch.no_grad():
    # TODO: Encode text features
    text_features = clip_model.encode_text(text_token)  # [1, embed_dim]
    text_features = F.normalize(text_features, dim=-1)  # [1, dim]
    for idx in layer_indices:
        # Get patch features from selected layer
        patch_features = get_patch_features(clip_model, image_tensor, block_idx=idx)  # [1, num_patches, dim]
        patch_features = F.normalize(patch_features, dim=-1)
        # TODO: Compute cosine similarity
        similarities = (patch_features @ text_features.unsqueeze(-1)).squeeze(-1)  # [1, num_patches]
        similarities = similarities.squeeze(0).cpu().numpy()  # [num_patches]
        
        # TODO: Reshape to spatial dimensions
        similarity_map = similarities.reshape(patch_grid_size, patch_grid_size)
        
        # Normalize
        similarity_map = (similarity_map - similarity_map.min()) / (similarity_map.max() - similarity_map.min() + 1e-8)
        
        # Resize to original image size
        similarity_heatmap_resized = cv2.resize(similarity_map, original_size, interpolation=cv2.INTER_LINEAR)
        similarity_heatmaps.append(similarity_heatmap_resized)

# Visualize all heatmaps together
fig, axes = plt.subplots(1, len(layer_indices) + 1, figsize=(6 * (len(layer_indices) + 1), 6), constrained_layout=True)

# Show original image first
axes[0].imshow(image, aspect='equal')
axes[0].set_title("Original Image")
axes[0].axis('off')

# Show similarity heatmaps (This is just for visualization, not used in the final report)
for i, heatmap in enumerate(similarity_heatmaps):
    ax = axes[i+1]
    ax.imshow(image, alpha=0.5, aspect='equal')
    im = ax.imshow(heatmap, cmap='jet', alpha=0.5, aspect='equal')
    ax.set_title(f"Cosine Sim. Heatmap\nLayer {layer_indices[i]}")
    ax.axis('off')
plt.colorbar(im, ax=ax)

plt.show()

print("Method 2: Feature Cosine Similarity completed for layers:", layer_indices)


## Method 3: Attention Map

In [ ]:
# TODO: Get attention map from the transformer block at block_idx
def get_attention_map(model, image_tensor, block_idx=11):
    """Extract attention maps from the transformer block at block_idx"""
    visual = model.visual
    
    # TODO: Prepare visual transformer input
    x = visual.conv1(image_tensor)  # [B, embed_dim, grid_h, grid_w]
    x = x.reshape(x.shape[0], x.shape[1], -1)  # [B, embed_dim, num_patches]
    x = x.permute(0, 2, 1)  # [B, num_patches, embed_dim]
    
    # Add CLS token
    cls_token = visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
    x = torch.cat([cls_token, x], dim=1)  # [B, 1 + num_patches, embed_dim]
    
    # Add positional embedding
    x = x + visual.positional_embedding.to(x.dtype)
    
    # Pre-LayerNorm
    x = visual.ln_pre(x)
    
    # Convert to [seq_len, B, embed_dim] for transformer
    x = x.permute(1, 0, 2)  # [seq_len, B, embed_dim]
    
    
    # TODO: Forward through the the block prior to the assigned block
    for i, resblock in enumerate(visual.transformer.resblocks):
        if i < block_idx:
            x = resblock(x)
        else:
            break
        
    # TODO: Get attention map for all patches from the assigned block
    target_block = visual.transformer.resblocks[block_idx]

    # Apply layer norm before attention
    x_norm = target_block.ln_1(x)  # [seq_len, B, embed_dim]
    
    # Get the attention module
    attn = target_block.attn

    # TODO: Compute Q, K, V
    # in_proj_weight: [3*embed_dim, embed_dim], in_proj_bias: [3*embed_dim]
    seq_len, batch_size, embed_dim = x_norm.shape
    num_heads = attn.num_heads
    head_dim = embed_dim // num_heads

    # Linear projection for Q, K, V
    qkv = F.linear(x_norm, attn.in_proj_weight, attn.in_proj_bias)  # [seq_len, B, 3*embed_dim]
    qkv = qkv.reshape(seq_len, batch_size, 3, num_heads, head_dim)
    qkv = qkv.permute(2, 1, 3, 0, 4)  # [3, B, num_heads, seq_len, head_dim]
    q, k, v = qkv[0], qkv[1], qkv[2]  # Each: [B, num_heads, seq_len, head_dim]

    # TODO: Compute attention scores
    scale = (head_dim ** -0.5)
    attn_weights = torch.matmul(q * scale, k.transpose(-2, -1))  # [B, num_heads, num_tokens, num_tokens]
    attn_weights = F.softmax(attn_weights, dim=-1)
    
    # TODO: Average over heads
    attn_weights = attn_weights.mean(dim=1)  # [B, seq_len, seq_len]
    
    return attn_weights

# Execute for multiple layers
original_size = image.size
attention_heatmaps = []
patch_indices = []
rect_coords = []

# Get attention maps for each layer
for block_idx in layer_indices:
    # Get most similar patch index once using one of the layers (typically the last)
    with torch.no_grad():
        # TODO: Encode text features
        text_features = clip_model.encode_text(text_token)  # [1, embed_dim]
        text_features = F.normalize(text_features, dim=-1)
        
        patch_features = get_patch_features(clip_model, image_tensor, block_idx=block_idx)
        patch_features = F.normalize(patch_features, dim=-1)
        
        # TODO: Find most similar patch
        similarities = (patch_features @ text_features.unsqueeze(-1)).squeeze(-1).squeeze(0)  # [num_patches]
        most_similar_patch_idx = similarities.argmax().item()
        print(f"Most similar patch index (using layer {block_idx}): {most_similar_patch_idx}")


        attention_weights = get_attention_map(clip_model, image_tensor, block_idx=block_idx)  # [B, num_tokens, num_tokens]
        full_attention_map = attention_weights[0].detach().cpu().numpy()  # [num_tokens, num_tokens]
    
    # TODO: Get attention from selected patch to all patches (exclude CLS token)
    attention_from_selected = full_attention_map[most_similar_patch_idx + 1, 1:]  # [num_patches]
    
    # TODO: Reshape to spatial dimensions
    attention_map = attention_from_selected.reshape(patch_grid_size, patch_grid_size)
    # Normalize
    attention_map = (attention_map - attention_map.min()) / (attention_map.max() - attention_map.min() + 1e-8)
    # Resize to original image size
    attention_heatmap_resized = cv2.resize(attention_map, original_size, interpolation=cv2.INTER_LINEAR)
    attention_heatmaps.append(attention_heatmap_resized)

    # Calculate red box position for the most similar patch
    patch_row = most_similar_patch_idx // patch_grid_size
    patch_col = most_similar_patch_idx % patch_grid_size
    img_w, img_h = original_size
    patch_w = img_w / patch_grid_size
    patch_h = img_h / patch_grid_size
    rect_x = patch_col * patch_w
    rect_y = patch_row * patch_h
    rect_coords.append((rect_x, rect_y, patch_w, patch_h))

# Visualize all the attention heatmaps together (This is just for visualization, not used in the final report)
fig, axes = plt.subplots(1, len(layer_indices) + 1, figsize=(6 * (len(layer_indices) + 1), 6), constrained_layout=True)
# Original image
axes[0].imshow(image, aspect='equal')
axes[0].set_title("Original Image")
axes[0].axis('off')

for i, (heatmap, block_idx) in enumerate(zip(attention_heatmaps, layer_indices)):
    ax = axes[i + 1]
    im = ax.imshow(heatmap, cmap='jet', alpha=0.5, aspect='equal')
    ax.imshow(image, alpha=0.5, aspect='equal')
    ax.set_title(f"Attention Map\nLayer {block_idx}")
    ax.axis('off')
    # Draw red rectangle on the most similar patch
    rect_x, rect_y, patch_w, patch_h = rect_coords[i]
    rect = patches.Rectangle(
        (rect_x, rect_y), patch_w, patch_h,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
plt.colorbar(im, ax=ax)

plt.show()
print(f"Method 3: Attention Map with Selected Patch completed for layers: {layer_indices}")


NameError: name 'image' is not defined

## Method 4: CLIP Surgery

In [ ]:
# import third_party/CLIP_Surgery/clip.py
# Import CLIP surgery's clip module for model and feature surgery functions
from third_party.CLIP_Surgery.clip import clip
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms import InterpolationMode
BICUBIC = InterpolationMode.BICUBIC

In [ ]:
# CLIP Surgery 
model, preprocess = clip.load("CS-ViT-B/16", device=device)
model.eval()

# Preprocess image with 512x512 for higher resolution
preprocess_surgery = Compose([Resize((512, 512), interpolation=BICUBIC), ToTensor(),
    Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))])

# Load and preprocess image
cv2_img = cv2.imread(img_dict[data_index])
cv2_img = cv2.resize(cv2_img, (512, 512))
pil_img = Image.open(img_dict[data_index]).convert('RGB')
image_input = preprocess_surgery(pil_img).unsqueeze(0).to(device)

# Get text prompt - IMPORTANT: need multiple prompts for CLIP Surgery to work!
text_prompt = basic_caption_dict[data_index]
# Add background prompts to make feature surgery work
texts = [text_prompt, "background"]  
text_tokens = clip.tokenize(texts).to(device)

# Run CLIP Surgery inference
with torch.no_grad():
    # Encode image features
    image_features = model.encode_image(image_input)
    
    # Encode text features for multiple prompts
    text_features = model.encode_text(text_tokens)
    
    # Normalize features
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    
    # Apply CLIP feature surgery
    similarity = clip.clip_feature_surgery(image_features, text_features)
    
    # Debug
    print(f"similarity shape: {similarity.shape}")
    print(f"similarity - min: {similarity.min():.4f}, max: {similarity.max():.4f}")
    
    # Get similarity map using official function
    similarity_map = clip.get_similarity_map(similarity[:, 1:, :], cv2_img.shape[:2])
    
    # Extract the similarity map for TARGET text (index 0, not background)
    similarity_map = similarity_map[0, :, :, 0].cpu().numpy()

print(f"similarity_map - min: {similarity_map.min():.4f}, max: {similarity_map.max():.4f}")

# Resize to original image size
original_size = image.size
clip_surgery_heatmap_resized = cv2.resize(similarity_map, original_size, interpolation=cv2.INTER_LINEAR)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

axes[0].imshow(image, aspect='equal')
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(image, aspect='equal')
im = axes[1].imshow(clip_surgery_heatmap_resized, cmap='jet', alpha=0.5, aspect='equal')
axes[1].set_title("CLIP Surgery Heatmap")
axes[1].axis('off')

plt.colorbar(im, ax=axes[1])
plt.show()

print("Method 4: CLIP Surgery completed!")

## Method 5: GEM (Grounding Everything Module)


In [ ]:
import gem

In [ ]:
# Get GEM's image preprocessing transform
preprocess_gem = gem.get_gem_img_transform()

# Create GEM model
model_name = 'ViT-B/16'
pretrained = 'openai'
gem_model = gem.create_gem_model(model_name=model_name, pretrained=pretrained, device=device)

# Prepare image using GEM's preprocess
image_tensor_gem = preprocess_gem(image).unsqueeze(0).to(device)

# Get text prompt
text_prompt = basic_caption_dict[data_index]

# Run GEM inference
with torch.no_grad():
    logits = gem_model(image_tensor_gem, [text_prompt])  # [B, num_prompt, W, H]

# Extract heatmap (first batch, first prompt)
heatmap_gem = logits[0, 0].cpu().numpy()  # [W, H]

# Normalize to [0, 1]
heatmap_gem = (heatmap_gem - heatmap_gem.min()) / (heatmap_gem.max() - heatmap_gem.min() + 1e-8)

# Resize to original image size if needed
original_size = image.size
if heatmap_gem.shape != original_size[::-1]:  # PIL size is (W, H), numpy is (H, W)
    heatmap_gem_resized = cv2.resize(heatmap_gem, original_size, interpolation=cv2.INTER_LINEAR)
else:
    heatmap_gem_resized = heatmap_gem

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

axes[0].imshow(image, aspect='equal')
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(image, alpha=0.5, aspect='equal')
im = axes[1].imshow(heatmap_gem_resized, cmap='jet', alpha=0.5, aspect='equal')
axes[1].set_title("GEM Heatmap")
axes[1].axis('off')

plt.colorbar(im, ax=axes[1])
plt.show()

print("Method 5: GEM completed!")

## Visulize all images for submission

In [ ]:
# Visualize all images from probing methods, ensuring all heatmaps match the original image size

def show_original(ax, title="Original Image"):
    ax.imshow(image, aspect='equal')
    ax.set_title(title)
    ax.axis('off')

def show_heatmaps_row(ax_row, heatmaps, titles):
    """
    Overlay each heatmap in one row.
    Args:
        ax_row: list of matplotlib axes in this row.
        heatmaps: list of heatmaps to show.
        titles: list of titles for each heatmap.
    """
    for ax, heatmap, title in zip(ax_row, heatmaps, titles):
        ax.imshow(image, alpha=0.5, aspect='equal')
        im = ax.imshow(heatmap, cmap='jet', alpha=0.5, aspect='equal')
        ax.set_title(title)
        ax.axis('off')

# Prepare heatmaps and titles for each row
gradcam_titles = [f"GradCAM Heatmap\nLayer {idx}" for idx in layer_indices]
cosine_titles = [f"Cosine Sim. Heatmap\nLayer {idx}" for idx in layer_indices]
attention_titles = [f"Attention Map\nLayer {idx}" for idx in layer_indices]

other_heatmaps = [
    ("CLIP Surgery", clip_surgery_heatmap_resized),
    ("GEM", heatmap_gem_resized),
]

# Set up figure and axes: 4 rows, 5 columns
fig, axes = plt.subplots(4, 5, figsize=(30, 16), constrained_layout=True)

# Row 1: GradCAM heatmaps
show_original(axes[0][0])
show_heatmaps_row(
    axes[0, 1:],
    gradcam_heatmaps_resized,
    gradcam_titles
)

# Row 2: Cosine Similarity heatmaps
show_original(axes[1][0])
show_heatmaps_row(
    axes[1, 1:],
    similarity_heatmaps,
    cosine_titles
)

# Row 3: Attention Maps
show_original(axes[2][0])
show_heatmaps_row(
    axes[2, 1:],
    attention_heatmaps,
    attention_titles
)

# Row 4: Other methods (CLIP Surgery, GEM)
show_original(axes[3][0])
other_titles, other_maps = zip(*other_heatmaps)
show_heatmaps_row(
    axes[3, 1:1 + len(other_maps)],
    other_maps,
    other_titles
)
# Hide unused subplots in the last row if any
for ax in axes[3, 1 + len(other_maps):]:
    ax.axis('off')

plt.show()

# Save the figure (This will be used in the final report)
# TODO: No code editing needed, please submit the figure in the final report
image_name = img_dict[data_index].split("/")[-1].split(".")[0]
fig.savefig(f"./results/{image_name}_final.png", dpi=180)
